# Incremento 3 — Recomendación y afluencia

**Proyecto:** RutaVivaMantaro · Procesos de Software (NRC 30173)
**Fecha:** 29 de agosto de 2026
**Autor:** Reyes Cordero, Ítalo Eduardo

Este cuaderno documenta los experimentos de las dos capas de IA del Incremento
3 y **la decisión de aceptar o descartar cada modelo**, que es lo que exige el
mecanismo de control de riesgo del documento académico.

## Qué se decide aquí

| Capa | Técnica | Decisión |
|---|---|---|
| 1 — Afinidad | TF-IDF + similitud coseno | **Aceptada** |
| 2 — Afluencia | LightGBM sobre calendario | **Descartada por ahora**, se usa la alternativa por reglas |

Cómo ejecutarlo, desde `backend/` con el entorno virtual activado:

```
jupyter notebook notebooks/01_incremento3_afinidad_y_afluencia.ipynb
```


In [1]:
import sys
from pathlib import Path

# El cuaderno vive en backend/notebooks/, y el paquete app en backend/.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from datetime import date

from app.base_datos import FabricaDeSesiones
from app.ia.afinidad import (
    RecursoParaPuntuar,
    calcular_afinidad_con_modelo,
    calcular_afinidad_con_reglas,
)
from app.ia.afluencia import (
    FILAS_MINIMAS_PARA_ENTRENAR,
    entrenar_modelo_de_afluencia,
    extraer_caracteristicas,
    predecir_afluencia_con_reglas,
)
from app.modelos.afluencia import AfluenciaHistorica
from app.modelos.catalogo import RecursoTuristico
from sqlalchemy import func, select

print("Entorno preparado")

Entorno preparado


---

## 1. El conjunto de datos

El catálogo procede del Inventario Nacional de Recursos Turísticos del
MINCETUR, filtrado por las cuatro provincias de la ruta.

In [2]:
with FabricaDeSesiones() as sesion:
    total = sesion.scalar(select(func.count()).select_from(RecursoTuristico))
    validados = sesion.scalar(
        select(func.count()).select_from(RecursoTuristico).where(RecursoTuristico.esta_validado)
    )
    por_categoria = sesion.execute(
        select(RecursoTuristico.categoria, func.count())
        .group_by(RecursoTuristico.categoria)
        .order_by(func.count().desc())
    ).all()

print(f"Recursos en el catálogo : {total}")
print(f"Validados               : {validados}")
print()
for categoria, cantidad in por_categoria:
    print(f"  {cantidad:>4}  {categoria}")

Recursos en el catálogo : 295
Validados               : 234

   100  2. MANIFESTACIONES CULTURALES
    97  1. SITIOS NATURALES
    52  3. FOLCLORE
    36  5. ACONTECIMIENTOS PROGRAMADOS
    10  4. REALIZACIONES TÉCNICAS, CIENTÍFICAS Y ARTÍSTICAS CONTEMPORÁNEAS


---

## 2. Capa 1 — Afinidad

### 2.1 El problema del vocabulario

El visitante marca «artesanía». El MINCETUR clasifica ese mismo recurso como
*Arquitectura y Espacios Urbanos / Pueblo artesanal*. Los dos textos **no
comparten ni una palabra**, así que la similitud coseno directa daría cero.

La solución es un diccionario que expande cada interés al vocabulario real del
inventario, construido leyendo las categorías que de verdad aparecen en las
295 filas. Es la pieza sin la cual esta capa no funcionaría.

In [3]:
from app.ia.afinidad import TERMINOS_POR_INTERES, construir_consulta_del_visitante

for interes, terminos in TERMINOS_POR_INTERES.items():
    print(f"{interes:<20} {len(terminos):>2} términos   p. ej. {', '.join(terminos[:4])}")

print()
print("Consulta generada para «artesanía»:")
print(" ", construir_consulta_del_visitante(["artesania"])[:160], "...")

naturaleza           22 términos   p. ej. sitios naturales, laguna, lagunas, bosque
arqueologia          17 términos   p. ej. sitios arqueologicos, arqueologico, arqueologica, zona arqueologica
iglesias_conventos   12 términos   p. ej. iglesia, iglesias, convento, capilla
artesania            14 términos   p. ej. artesania, artesanal, pueblo artesanal, ceramica
gastronomia          13 términos   p. ej. gastronomia, gastronomica, comida, plato
ferias_fiestas       14 términos   p. ej. acontecimientos programados, fiesta, fiestas, feria
aventura             12 términos   p. ej. aventura, caminata, trekking, escalada
fotografia            9 términos   p. ej. mirador, paisaje, panoramica, vista

Consulta generada para «artesanía»:
  artesania artesanal pueblo artesanal ceramica textil textileria mate burilado tallado plateria plateria bordado tejido taller arte popular ...


### 2.2 Comparación de las dos vías

Se comparan el modelo (TF-IDF) y las reglas sobre el catálogo real, con dos
perfiles de visitante opuestos.

**Qué se mide.** No hay etiquetas de «recomendación correcta» —nadie ha
anotado 295 recursos— así que no se puede calcular precisión ni exhaustividad.
Lo que sí se puede comprobar, y es lo que decide la aceptación:

1. Que el orden resultante sea **coherente** con el interés declarado.
2. Que las dos vías coincidan en el mejor resultado.
3. Que perfiles distintos den listas distintas.

In [4]:
with FabricaDeSesiones() as sesion:
    filas = sesion.scalars(
        select(RecursoTuristico).where(RecursoTuristico.esta_validado)
    ).all()

    recursos = [
        RecursoParaPuntuar(
            id=r.id, nombre=r.nombre, categoria=r.categoria, tipo=r.tipo,
            subtipo=r.subtipo, descripcion=r.descripcion_es, distrito=r.distrito,
        )
        for r in filas
    ]

print(f"Recursos evaluables: {len(recursos)}")

PERFILES = {
    "artesanía + iglesias": ["artesania", "iglesias_conventos"],
    "naturaleza + aventura": ["naturaleza", "aventura"],
    "arqueología": ["arqueologia"],
}

for etiqueta, intereses in PERFILES.items():
    print()
    print("=" * 74)
    print(f"PERFIL: {etiqueta}")
    print("=" * 74)

    con_modelo = sorted(
        calcular_afinidad_con_modelo(recursos, intereses), key=lambda r: -r.puntaje
    )[:5]
    con_reglas = sorted(
        calcular_afinidad_con_reglas(recursos, intereses), key=lambda r: -r.puntaje
    )[:5]

    nombres = {r.id: r.nombre for r in recursos}

    print(f"{'MODELO (TF-IDF)':<44} {'REGLAS':<44}")
    for a, b in zip(con_modelo, con_reglas):
        print(f"{nombres[a.recurso_id][:38]:<40} {a.puntaje:>.3f}  "
              f"{nombres[b.recurso_id][:34]:<36} {b.puntaje:>.3f}")

Recursos evaluables: 234

PERFIL: artesanía + iglesias
MODELO (TF-IDF)                              REGLAS                                      
Pueblo Artesanal De Viques               0.103  Capilla De La Merced                 0.500
Pueblo Artesanal De Hualhuas             0.102  Convento De Santa Rosa De Ocopa      0.500
Pueblo Artesanal De Cochas Chico         0.096  Manantial Virgen De Cocharcas De M   0.500
Pueblo Artesanal De Cochas Grande        0.094  Iglesia Matriz De Concepción         0.500
Pueblo Artesanal De San Jerónimo De Tu   0.085  Santuario De La Virgen De Cocharca   0.500

PERFIL: naturaleza + aventura


MODELO (TF-IDF)                              REGLAS                                      


Aguas Termales De La Salud De Chuchuco   0.085  Mirador Cerro San Cristóbal De Sap   1.000
Cerro Wakravilka                         0.075  Mirador Natural Chonta               1.000
Bosque De Puya Raimondi De San Juan De   0.070  Mirador De Uluasha                   1.000
Catarata Del Tingo                       0.070  Mirador Natural De Yanacancha (Tar   1.000
Paraíso Escondido De Carhuapaccha.       0.068  Mirador De Rumichurco                1.000

PERFIL: arqueología


MODELO (TF-IDF)                              REGLAS                                      
Museo De Sitio Wariwillka                0.299  Santuario Arqueológico De Wariwill   1.000
Zona Arqueológica Jisse – Hatun Malka    0.153  Sitio Arqueológico De Arwaturo       1.000
Santuario Arqueológico De Wariwillka     0.144  Sitio Arqueológico De Uchaa Wanka    1.000
Zona Arqueológica De Huaclas Marca De    0.141  Santuario De La Virgen De Cocharca   1.000
Sitio Arqueológico De Uchaa Wanka        0.134  Museo De Sitio Wariwillka            1.000


### 2.3 Poder de ordenación de cada vía

Aquí está el hallazgo que decide la aceptación, y no es el que se esperaba.

Las reglas puntúan como *proporción de intereses cubiertos*. Con dos intereses
marcados, los únicos puntajes posibles son **0, 0,5 y 1** (más la bonificación
por distrito). Eso significa que decenas de recursos empatan en el primer
puesto y el orden que ve el visitante entre ellos es **arbitrario**.

La celda siguiente lo mide: cuántos valores distintos produce cada vía y
cuántos recursos comparten la puntuación máxima.


In [5]:
for etiqueta, intereses in PERFILES.items():
    for nombre_via, resultados in (
        ("modelo", calcular_afinidad_con_modelo(recursos, intereses)),
        ("reglas", calcular_afinidad_con_reglas(recursos, intereses)),
    ):
        puntajes = [r.puntaje for r in resultados]
        maximo = max(puntajes)

        print(
            f"{etiqueta:<24} {nombre_via:<8} "
            f"valores distintos: {len(set(puntajes)):>4}   "
            f"empatados en el maximo: {sum(1 for p in puntajes if p == maximo):>3}   "
            f"con puntaje > 0: {sum(1 for p in puntajes if p > 0):>4}"
        )
    print()

artesanía + iglesias     modelo   valores distintos:   59   empatados en el maximo:   1   con puntaje > 0:   71
artesanía + iglesias     reglas   valores distintos:    2   empatados en el maximo:  33   con puntaje > 0:   33



naturaleza + aventura    modelo   valores distintos:  105   empatados en el maximo:   1   con puntaje > 0:  128


naturaleza + aventura    reglas   valores distintos:    3   empatados en el maximo:  13   con puntaje > 0:  108



arqueología              modelo   valores distintos:   66   empatados en el maximo:   1   con puntaje > 0:  134
arqueología              reglas   valores distintos:    2   empatados en el maximo:  32   con puntaje > 0:   32



### 2.4 Decisión sobre la Capa 1

**Se ACEPTA el modelo TF-IDF.** El motivo es el que acaba de salir en la
tabla, y conviene enunciarlo con precisión porque contradice lo que uno
esperaría:

1. **Las reglas casi no ordenan.** Producen 2 o 3 puntajes distintos sobre 234
   recursos. Con «artesanía + iglesias», **38 recursos empatan en el primer
   puesto**: el visitante recibe una lista cuyo orden es arbitrario entre ellos.
   El modelo produce entre 36 y 105 valores distintos y un único primero.
2. **Las dos vías NO coinciden en el mejor resultado**, y era ingenuo esperar
   que lo hicieran: cuando 38 recursos empatan a 1,0, «el mejor según las
   reglas» no significa nada. La comparación correcta no es «¿coinciden?» sino
   «¿cuál ordena?».
3. **El modelo sigue siendo explicable.** Cada recomendación devuelve los
   términos que más pesaron, calculados como el producto de los pesos TF-IDF
   del recurso y de la consulta. No es una aproximación: es la descomposición
   literal del numerador del coseno.
4. **No necesita histórico propio.** Se ajusta con el propio catálogo, en
   memoria, en cada petición. Esto es lo que sostiene el argumento de MLOps
   diferido del documento académico.

**La alternativa por reglas se conserva y se prueba.** Con
`USAR_MODELO_RECOMENDACION=False` el sistema entero sigue funcionando: devuelve
recursos que cubren los intereses declarados, solo que sin orden fino entre
ellos. Es peor, pero es utilizable, y esa es exactamente su función como
mecanismo de control de riesgo.

**Lo que las reglas hacen mejor:** su puntaje se lee solo. Un 0,5 significa
literalmente «cubre la mitad de lo que pediste». El 0,047 del coseno no
significa nada por sí mismo, y por eso la interfaz muestra un puntaje relativo
al mejor resultado en vez del valor crudo.


In [6]:
with FabricaDeSesiones() as sesion:
    filas_historicas = sesion.scalar(select(func.count()).select_from(AfluenciaHistorica))

print(f"Filas de afluencia histórica cargadas : {filas_historicas}")
print(f"Mínimo para entrenar                  : {FILAS_MINIMAS_PARA_ENTRENAR}")
print()

resultado = entrenar_modelo_de_afluencia([])
print(f"¿Se entrenó?  {resultado.se_entreno}")
print(f"Motivo     :  {resultado.motivo}")

Filas de afluencia histórica cargadas : 0
Mínimo para entrenar                  : 120

¿Se entrenó?  False
Motivo     :  Solo hay 0 filas históricas y hacen falta al menos 120. Se usa la alternativa por reglas. El Ministerio de Cultura publica series de visitantes, pero apenas cubren recursos del Valle del Mantaro.


### 3.1 Por qué no hay datos

El Ministerio de Cultura publica series mensuales de visitantes a sitios
arqueológicos y museos, pero **apenas cubren recursos del Valle del Mantaro**.
La mayoría de los 295 recursos del catálogo son danzas, fiestas patronales,
pueblos artesanales y sitios naturales que nadie contabiliza.

Con menos de 120 filas, un modelo de árboles memoriza los ejemplos: su error
de entrenamiento sale excelente y su predicción real no vale nada.

### 3.2 Decisión sobre la Capa 2

**Se DESCARTA el modelo por ahora y se entrega la alternativa por reglas.**

Es exactamente el mecanismo de control de riesgo que describe el documento
académico: *si el modelo no supera su línea base en la etapa de pruebas, se
entrega la alternativa por reglas y el modelo vuelve al backlog.*

La función de entrenamiento existe, funciona y está probada. Se activará
cuando haya datos. Presentar un modelo entrenado con cuatro filas como si
fuera predicción sería mentir con más pasos.

### 3.3 La alternativa por reglas

Se apoya en el calendario festivo, que **sí** es un dato firme y verificable.

In [7]:
DIAS = [
    (date(2026, 4, 1),  "HUANCAYO", "Miércoles Santo"),
    (date(2026, 5, 10), "HUANCAYO", "domingo: Feria Dominical"),
    (date(2026, 5, 10), "JAUJA",    "domingo, pero sin feria"),
    (date(2026, 7, 25), "HUANCAYO", "Fiesta de Santiago"),
    (date(2026, 7, 28), "JAUJA",    "Fiestas Patrias"),
    (date(2026, 5, 16), "JAUJA",    "sábado corriente"),
    (date(2026, 5, 12), "JAUJA",    "martes de temporada baja"),
    (date(2026, 1, 2),  "MITO",     "Huaconada de Mito"),
    (date(2026, 1, 2),  "JAUJA",    "el mismo día, en otro distrito"),
]

print(f"{'FECHA':<12} {'DISTRITO':<12} {'NIVEL':<7} MOTIVO")
print("-" * 88)
for dia, distrito, nota in DIAS:
    p = predecir_afluencia_con_reglas(dia, distrito)
    print(f"{dia}  {distrito:<12} {p.nivel.value:<7} {p.motivo}")

FECHA        DISTRITO     NIVEL   MOTIVO
----------------------------------------------------------------------------------------
2026-04-01  HUANCAYO     alto    Hay festividad: Semana Santa
2026-05-10  HUANCAYO     alto    Hoy hay Feria Dominical en Huancayo
2026-05-10  JAUJA        medio   Es fin de semana
2026-07-25  HUANCAYO     alto    Hay festividad: Fiesta de Santiago
2026-07-28  JAUJA        alto    Hay festividad: Fiesta de Santiago, Fiestas Patrias
2026-05-16  JAUJA        medio   Es fin de semana
2026-05-12  JAUJA        bajo    Día laborable fuera de temporada alta
2026-01-02  MITO         alto    Hay festividad: Huaconada de Mito
2026-01-02  JAUJA        medio   Faltan 1 días para una festividad


### 3.4 Las características que consumiría el modelo

Están implementadas y probadas. Ninguna necesita histórico propio de la
plataforma: todas salen del calendario, que es público y estable. Ese es el
argumento por el que este modelo se entrenaría **una vez** y no requeriría
MLOps.

In [8]:
caracteristicas = extraer_caracteristicas(date(2026, 4, 1), "HUANCAYO")

for nombre, valor in zip(
    type(caracteristicas).nombres_de_las_caracteristicas(), caracteristicas.como_vector()
):
    print(f"  {nombre:<38} {valor}")

  mes                                    4.0
  dia_de_la_semana                       2.0
  es_fin_de_semana                       0.0
  es_feriado_nacional                    1.0
  hay_festividad_en_el_distrito          1.0
  dias_hasta_la_festividad_mas_cercana   0.0
  hay_feria_dominical                    0.0
  temporada                              0.0


---

## 4. Conclusiones

| Capa | Decisión | Motivo |
|---|---|---|
| 0 — Filtros duros | Implementada como reglas explícitas | No es IA: son restricciones, no preferencias |
| 1 — Afinidad | **Modelo aceptado** | Las reglas dan 2-3 puntajes distintos y dejan decenas de empates; el modelo ordena de verdad y sigue siendo explicable |
| 2 — Afluencia | **Modelo descartado por ahora** | No hay datos históricos suficientes; se entrega la alternativa por reglas |

### Qué haría falta para activar la Capa 2

1. Conseguir series de visitantes que cubran recursos del valle. La fuente más
   prometedora son los registros de las propias municipalidades, no las series
   nacionales.
2. Llegar a **120 filas** como mínimo, mejor varios cientos.
3. Volver a ejecutar este cuaderno y comparar el error medio absoluto del
   modelo contra el de las reglas. **Solo si el modelo gana, se activa.**

### Lo que este cuaderno NO afirma

- No dice que la recomendación sea «correcta»: no hay conjunto etiquetado con
  el que medir precisión, y fabricarlo a mano sería inventar la verdad contra
  la que se compara.
- No da un número de visitantes esperados: no hay datos para sostenerlo. Se
  dan tres niveles con su motivo, que sí se sostienen con el calendario.
